# updatesupport-finance: Portfolio Model-Risk Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nahuaque/updatesupport/blob/main/packages/updatesupport-finance/examples/notebooks/model_risk_portfolio_colab.ipynb)

This notebook is a tutorial for using `updatesupport-finance` as a model-risk review aid.

We will audit a synthetic retail-credit report that publishes expected loss by `product`, `region`, `fico_band`, and `ltv_band`. The question is whether that public segmentation still supports the reported expected-loss rate if hidden portfolio mix changes inside each public bucket.

By the end, you should be able to:

- read the public and hidden cell setup,
- interpret hidden-composition ambiguity as separate from model error,
- identify which public fibers drive instability,
- use refinement and frontier plots to decide what extra fields are worth reporting.

## 1. Install and import

Run the install cell once in Colab. It installs the core package with the finance extra, plus plotting and widget dependencies.

If you are running from a local source checkout, you can skip the install cell and use your existing environment.

In [ ]:
%pip install -q --upgrade "updatesupport[finance,cvxpy]>=0.1.5" "updatesupport-finance>=0.1.4" pandas seaborn matplotlib ipywidgets

try:
    from google.colab import output

    output.enable_custom_widget_manager()
except Exception:
    pass

In [ ]:
import io
import textwrap
from importlib.metadata import version

import cvxpy as cp
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

import updatesupport as us
import updatesupport_finance as usf

print(f"updatesupport {version('updatesupport')}")
print(f"updatesupport-finance {version('updatesupport-finance')}")
print(f"cvxpy {version('cvxpy')}; solvers: {', '.join(cp.installed_solvers())}")

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="deep",
    rc={"axes.spines.right": False, "axes.spines.top": False},
)

## 2. Synthetic retail-credit portfolio

Each row is a retained hidden portfolio cell. The public report only shows `product x region x fico_band x ltv_band`, while internal monitoring data also tracks broker channel, employment type, vintage, hardship history, documentation type, local housing market, and borrower cashflow pattern.

The target metric is exposure-weighted expected loss: `pd * lgd`, weighted by `ead`. In a real review, these rows would usually come from a model-monitoring or validation extract.

In [ ]:
PUBLIC_COLUMNS = ("product", "region", "fico_band", "ltv_band")
HIDDEN_COLUMNS = (
    "product",
    "region",
    "fico_band",
    "ltv_band",
    "broker_channel",
    "employment_type",
    "vintage",
    "hardship_history",
    "documentation_type",
    "local_housing_market",
    "cashflow_pattern",
)
CANDIDATE_REFINEMENTS = (
    "broker_channel",
    "employment_type",
    "vintage",
    "hardship_history",
    "documentation_type",
    "local_housing_market",
    "cashflow_pattern",
)
FACTOR_COLUMNS = {
    "macro_beta": "macro_beta",
    "rate_sensitivity": "rate_sensitivity",
    "housing_beta": "housing_beta",
}

portfolio_csv = """
account_count,product,region,fico_band,ltv_band,broker_channel,employment_type,vintage,hardship_history,documentation_type,local_housing_market,cashflow_pattern,pd,lgd,ead
140,mortgage,north,prime,low,broker,salaried,2024,none,full_doc,stable,stable,0.014,0.32,15400000
90,mortgage,north,prime,low,direct,self_employed,2023,prior,alt_doc,cooling,seasonal,0.038,0.46,8100000
110,mortgage,north,prime,medium,broker,salaried,2022,none,full_doc,cooling,stable,0.023,0.39,12650000
70,mortgage,north,prime,medium,correspondent,contractor,2021,prior,full_doc,declining,volatile,0.049,0.52,7700000
85,mortgage,south,near_prime,high,broker,salaried,2024,none,full_doc,stable,stable,0.058,0.56,7225000
65,mortgage,south,near_prime,high,correspondent,self_employed,2022,current,alt_doc,declining,volatile,0.118,0.67,5525000
160,auto,west,prime,medium,dealer,salaried,2024,none,full_doc,stable,stable,0.031,0.50,3840000
100,auto,west,prime,medium,online,contractor,2023,prior,stated_income,stable,seasonal,0.061,0.58,2400000
190,card,east,near_prime,na,direct,salaried,2024,none,full_doc,stable,stable,0.074,0.82,1900000
120,card,east,near_prime,na,affiliate,contractor,2022,current,thin_file,cooling,volatile,0.132,0.90,1200000
"""

portfolio = pd.read_csv(io.StringIO(textwrap.dedent(portfolio_csv).strip()))
portfolio["expected_loss_rate"] = portfolio["pd"] * portfolio["lgd"]
portfolio["expected_loss_amount"] = portfolio["expected_loss_rate"] * portfolio["ead"]
portfolio["macro_beta"] = (
    portfolio["product"].map({"mortgage": 1.10, "auto": 0.80, "card": 1.35})
    + portfolio["fico_band"].map({"prime": -0.10, "near_prime": 0.18})
    + portfolio["cashflow_pattern"].map(
        {"stable": -0.05, "seasonal": 0.06, "volatile": 0.16}
    )
).round(3)
portfolio["rate_sensitivity"] = (
    portfolio["product"].map({"mortgage": 1.25, "auto": 0.55, "card": 0.25})
    + portfolio["ltv_band"].map({"low": -0.15, "medium": 0.05, "high": 0.24, "na": 0.0})
    + portfolio["vintage"].map({2021: 0.16, 2022: 0.10, 2023: 0.04, 2024: -0.03})
).round(3)
portfolio["housing_beta"] = (
    portfolio["product"].map({"mortgage": 1.30, "auto": 0.20, "card": 0.10})
    + portfolio["local_housing_market"].map(
        {"stable": -0.05, "cooling": 0.12, "declining": 0.28}
    )
).round(3)
portfolio["public_segment"] = (
    portfolio.loc[:, PUBLIC_COLUMNS].astype(str).agg(" | ".join, axis=1)
)
portfolio["hidden_label"] = (
    portfolio["broker_channel"]
    + " / "
    + portfolio["employment_type"]
    + " / "
    + portfolio["vintage"].astype(str)
)
portfolio

## 3. Inspect exposure and hidden risk variation

Start with the familiar descriptive view. The first chart shows where exposure sits across public segments. Large bars identify segments that can matter a lot to the portfolio-level result even if their expected-loss rate is not extreme.

The second chart shows the observed expected-loss rate for each public segment, with marker size indicating how many hidden cells sit underneath it. A large marker means the published bucket is hiding multiple internal cells, so the public rate may depend on the hidden mix.

Then use the dropdown to inspect one public bucket at a time. This is the key intuition: the public segment can look coherent in aggregate while hiding materially different risk cells inside the same reported bucket.

In [ ]:
public_summary = (
    portfolio.groupby(
        ["product", "region", "fico_band", "ltv_band", "public_segment"], as_index=False
    )
    .agg(
        ead=("ead", "sum"),
        expected_loss_amount=("expected_loss_amount", "sum"),
        hidden_cells=("expected_loss_rate", "size"),
    )
    .assign(
        ead_millions=lambda data: data["ead"] / 1_000_000,
        expected_loss_rate=lambda data: data["expected_loss_amount"] / data["ead"],
    )
    .sort_values("ead_millions", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
sns.barplot(
    data=public_summary, y="public_segment", x="ead_millions", hue="product", ax=axes[0]
)
axes[0].set_title("Exposure by public segment")
axes[0].set_xlabel("EAD, $mm")
axes[0].set_ylabel("")
sns.scatterplot(
    data=public_summary,
    x="ead_millions",
    y="expected_loss_rate",
    size="hidden_cells",
    hue="product",
    sizes=(80, 280),
    ax=axes[1],
)
axes[1].set_title("Public segment expected-loss rate")
axes[1].set_xlabel("EAD, $mm")
axes[1].set_ylabel("Expected-loss rate")
plt.show()

In [ ]:
segment_widget = widgets.Dropdown(
    options=sorted(portfolio["public_segment"].unique()),
    value=sorted(portfolio["public_segment"].unique())[0],
    description="Public cell",
    layout=widgets.Layout(width="720px"),
)


@widgets.interact(public_segment=segment_widget)
def hidden_cell_plot(public_segment: str):
    subset = portfolio.loc[portfolio["public_segment"] == public_segment].copy()
    subset = subset.sort_values("expected_loss_rate", ascending=False)
    fig, ax = plt.subplots(figsize=(12, max(3.5, 0.55 * len(subset))))
    sns.barplot(
        data=subset,
        y="hidden_label",
        x="expected_loss_rate",
        hue="cashflow_pattern",
        ax=ax,
    )
    ax.set_title("Hidden-cell risk variation inside one public bucket")
    ax.set_xlabel("Expected-loss rate")
    ax.set_ylabel("")
    ax.legend(title="Cashflow", loc="lower right")
    plt.show()

## 4. Run the finance model-risk report

Now we run the actual stability audit. The default stress is `saturated`: public segment masses stay fixed, and retained hidden cells inside each public segment may be rearranged arbitrarily. That is the assumption-light control question: does the public segmentation pin down the expected-loss metric at all on the retained support?

The bounded mix-shift option is a secondary sensitivity scenario. It uses `q_portfolio_mix_shift(radius=...)`, so the radius should be interpreted as a documented governance assumption rather than a universal standard.

The dashboard prints the reported expected-loss rate, the hidden-composition interval, the ambiguity width, and a review status. The interval is not a confidence interval. It is the range of values compatible with the fixed public distribution and the selected hidden-mix stress test.

The left plot shows which public fibers contribute most to ambiguity. A high contribution usually means the public bucket is both material and internally heterogeneous. The right plot ranks one-column refinements by how much ambiguity they remove. A strong refinement is a hidden field that explains much of the instability left unresolved by the public segmentation.


In [ ]:
def finance_q(stress: str, q_radius: float):
    if stress == "saturated":
        return "saturated"
    if stress == "bounded mix shift":
        return usf.q_portfolio_mix_shift(radius=q_radius)
    raise ValueError(f"unknown stress: {stress}")


def build_report(
    stress: str = "saturated",
    q_radius: float = 0.35,
    ambiguity_limit: float = 0.006,
):
    return usf.model_risk_report(
        portfolio.to_dict("records"),
        public=PUBLIC_COLUMNS,
        hidden=HIDDEN_COLUMNS,
        metric=usf.expected_loss(pd="pd", lgd="lgd"),
        exposure="ead",
        candidate_refinements=CANDIDATE_REFINEMENTS,
        q=finance_q(stress, q_radius),
        model_id="EL_SYNTHETIC_RETAIL_001",
        portfolio_name="Synthetic retail credit portfolio",
        as_of_date="2026-06-30",
        intended_use="Expected-loss segmentation model review",
        ambiguity_limit=ambiguity_limit,
        public_adequacy_required=False,
        top=5,
    )


def report_frames(report):
    fibers = pd.DataFrame(row.as_dict() for row in report.core.fibers)
    refinements = pd.DataFrame(row.as_dict() for row in report.core.refinements)
    return fibers, refinements


@widgets.interact(
    stress=widgets.Dropdown(
        options=("saturated", "bounded mix shift"),
        value="saturated",
        description="Stress",
    ),
    q_radius=widgets.FloatSlider(
        value=0.35, min=0.0, max=0.75, step=0.05, description="Q radius"
    ),
    ambiguity_limit=widgets.FloatSlider(
        value=0.006,
        min=0.0,
        max=0.025,
        step=0.001,
        readout_format=".3f",
        description="Limit",
    ),
)
def audit_dashboard(stress: str, q_radius: float, ambiguity_limit: float):
    report = build_report(
        stress=stress, q_radius=q_radius, ambiguity_limit=ambiguity_limit
    )
    print(f"stress: {stress}")
    print(f"reported expected-loss rate: {report.core.observed_value:.4%}")
    print(
        f"hidden-composition interval: [{report.core.interval.lower:.4%}, {report.core.interval.upper:.4%}]"
    )
    print(f"ambiguity width: {report.core.interval.diameter:.4%}")
    print(f"review status: {report.review_status}")
    fibers, refinements = report_frames(report)
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
    if not fibers.empty:
        fibers = fibers.assign(
            public_label=fibers["public_value"].map(
                lambda value: " | ".join(map(str, value))
            )
        )
        sns.barplot(
            data=fibers.sort_values("contribution", ascending=False).head(5),
            y="public_label",
            x="contribution",
            color="#4C72B0",
            ax=axes[0],
        )
        axes[0].set_title("Worst public fibers")
        axes[0].set_xlabel("Contribution to ambiguity")
        axes[0].set_ylabel("")
    if not refinements.empty:
        sns.barplot(
            data=refinements.sort_values("reduction", ascending=False).head(7),
            y="column",
            x="reduction",
            color="#55A868",
            ax=axes[1],
        )
        axes[1].set_title("Best one-column refinements")
        axes[1].set_xlabel("Ambiguity reduction")
        axes[1].set_ylabel("")
    plt.show()

## 5. CVXPY concentration-stress reports

The previous dashboard starts with saturated stress and optionally explores bounded cell movement. Finance reviews often want additional structured sensitivity checks: hidden mix can move, but portfolio-level factor exposure should not drift arbitrarily.

This section uses CVXPY-backed Q presets. The factor-exposure stress solves a conic optimization problem of the form `|| standardized_factor_shift ||_2 <= radius`. In model-risk language, this asks: if public buckets stay fixed and hidden composition shifts are constrained by portfolio factor exposure, how much can the expected-loss estimate move?

The dual-diagnostics chart shows the largest solver multipliers when CVXPY exposes them. Treat these as local sensitivity diagnostics, not as standalone validation evidence.


In [ ]:
def build_cvxpy_report(q, label: str, ambiguity_limit: float = 0.006):
    return usf.model_risk_report(
        portfolio.to_dict("records"),
        public=PUBLIC_COLUMNS,
        hidden=HIDDEN_COLUMNS,
        metric=usf.expected_loss(pd="pd", lgd="lgd"),
        exposure="ead",
        candidate_refinements=CANDIDATE_REFINEMENTS,
        q=q,
        model_id=f"EL_SYNTHETIC_RETAIL_001_{label.upper().replace(' ', '_')}",
        portfolio_name="Synthetic retail credit portfolio",
        as_of_date="2026-06-30",
        intended_use=f"Expected-loss segmentation review: {label}",
        ambiguity_limit=ambiguity_limit,
        public_adequacy_required=False,
        top=5,
    )


def cvxpy_scenario_frame(reports):
    rows = []
    for scenario, report in reports:
        stress = report.to_tables()["finance_concentration_stress"][0]
        rows.append(
            {
                "scenario": scenario,
                "lower": report.core.interval.lower,
                "upper": report.core.interval.upper,
                "ambiguity": report.core.interval.diameter,
                "review_status": report.review_status,
                "stress_type": stress["stress_type"],
                "moments": stress["moment_count"] or 0,
                "dual_rows": len(report.core.interval.duals),
            }
        )
    return pd.DataFrame(rows)


def cvxpy_dual_frame(reports, top: int = 8):
    rows = []
    for scenario, report in reports:
        for row in report.core.interval.dual_summary(top=top, min_magnitude=1e-8):
            rows.append(
                {
                    "scenario": scenario,
                    "constraint": f"{row.kind}: {row.name}",
                    "solve": row.solve,
                    "magnitude": row.magnitude,
                    "residual": row.residual,
                }
            )
    return pd.DataFrame(rows)


@widgets.interact(
    mix_radius=widgets.FloatSlider(
        value=0.35, min=0.0, max=0.75, step=0.05, description="Mix radius"
    ),
    tv_radius=widgets.FloatSlider(
        value=0.12, min=0.02, max=0.40, step=0.02, description="TV radius"
    ),
    factor_radius=widgets.FloatSlider(
        value=0.25, min=0.05, max=0.80, step=0.05, description="Factor radius"
    ),
    ambiguity_limit=widgets.FloatSlider(
        value=0.006,
        min=0.0,
        max=0.025,
        step=0.001,
        readout_format=".3f",
        description="Limit",
    ),
)
def cvxpy_stress_dashboard(
    mix_radius: float,
    tv_radius: float,
    factor_radius: float,
    ambiguity_limit: float,
):
    reports = (
        (
            "bounded cell shift",
            build_report(
                stress="bounded mix shift",
                q_radius=mix_radius,
                ambiguity_limit=ambiguity_limit,
            ),
        ),
        (
            "CVXPY TV budget",
            build_cvxpy_report(
                usf.q_exposure_weighted_tv(tv_radius, backend="cvxpy"),
                "CVXPY TV budget",
                ambiguity_limit=ambiguity_limit,
            ),
        ),
        (
            "CVXPY factor exposure",
            build_cvxpy_report(
                usf.q_factor_exposure_shift(
                    factor_radius,
                    portfolio.to_dict("records"),
                    hidden=HIDDEN_COLUMNS,
                    factors=FACTOR_COLUMNS,
                    exposure="ead",
                    backend="cvxpy",
                ),
                "CVXPY factor exposure",
                ambiguity_limit=ambiguity_limit,
            ),
        ),
    )
    scenarios = cvxpy_scenario_frame(reports)
    duals = cvxpy_dual_frame(reports)
    display(
        scenarios.style.format(
            {"lower": "{:.4%}", "upper": "{:.4%}", "ambiguity": "{:.4%}"}
        )
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
    palette = sns.color_palette("deep", len(scenarios))
    for y, (_, row), color in zip(range(len(scenarios)), scenarios.iterrows(), palette):
        axes[0].hlines(
            y=y,
            xmin=row["lower"],
            xmax=row["upper"],
            color=color,
            linewidth=7,
        )
        axes[0].scatter(
            [(row["lower"] + row["upper"]) / 2], [y], color="black", s=45, zorder=3
        )
    axes[0].set_yticks(range(len(scenarios)), scenarios["scenario"])
    axes[0].set_title("Stress-specific expected-loss intervals")
    axes[0].set_xlabel("Expected-loss rate")

    if duals.empty:
        axes[1].text(0.5, 0.5, "No CVXPY dual diagnostics", ha="center", va="center")
        axes[1].set_axis_off()
    else:
        dual_plot = duals.sort_values("magnitude", ascending=False).head(10)
        sns.barplot(
            data=dual_plot,
            y="constraint",
            x="magnitude",
            hue="scenario",
            ax=axes[1],
        )
        axes[1].set_title("Largest CVXPY dual diagnostics")
        axes[1].set_xlabel("Dual magnitude")
        axes[1].set_ylabel("")
    plt.show()

    if not duals.empty:
        display(duals.sort_values("magnitude", ascending=False).head(10))

## 6. Search the public-representation frontier

A reviewer rarely wants to publish every internal field. The frontier search asks a more practical question: which small public representation is stable enough?

Each point is one candidate public segmentation. The x-axis is reporting complexity, measured by public-cell count. The y-axis is worst-case ambiguity across the selected stress grid. This grid includes the saturated benchmark, the bounded mix-shift sensitivity scenario, and the observed no-shift baseline. Points below the dashed line meet the review limit.

The most useful answer is usually the smallest stable representation, not necessarily the representation with the absolute lowest ambiguity.


In [ ]:
def build_frontier(q_radius: float = 0.35, ambiguity_limit: float = 0.006):
    return us.public_representation_frontier(
        portfolio.to_dict("records"),
        base_public=PUBLIC_COLUMNS,
        hidden=HIDDEN_COLUMNS,
        target=usf.expected_loss(pd="pd", lgd="lgd"),
        weight="ead",
        candidate_refinements=CANDIDATE_REFINEMENTS,
        q_presets=("saturated", usf.q_portfolio_mix_shift(radius=q_radius), "observed"),
        min_cell_weights=(1.0,),
        ambiguity_limit=ambiguity_limit,
        bucket_budget=12,
        search="beam",
        beam_width=8,
        max_added_columns=3,
        max_evaluations=96,
        title="Synthetic Finance Public Representation Frontier",
    )


@widgets.interact(
    q_radius=widgets.FloatSlider(
        value=0.35, min=0.0, max=0.75, step=0.05, description="Q radius"
    ),
    ambiguity_limit=widgets.FloatSlider(
        value=0.006,
        min=0.0,
        max=0.025,
        step=0.001,
        readout_format=".3f",
        description="Limit",
    ),
)
def frontier_plot(q_radius: float, ambiguity_limit: float):
    frontier = build_frontier(q_radius=q_radius, ambiguity_limit=ambiguity_limit)
    candidate_rows = []
    for candidate in frontier.candidates:
        candidate_rows.append(
            {
                "label": candidate.label,
                "public_cells": candidate.public_cells,
                "max_ambiguity": candidate.max_ambiguity,
                "added_columns": candidate.added_column_count,
                "stable": bool(candidate.passes_ambiguity_limit),
                "frontier": candidate in frontier.frontier,
            }
        )
    data = pd.DataFrame(candidate_rows)
    fig, ax = plt.subplots(figsize=(11, 6))
    sns.scatterplot(
        data=data,
        x="public_cells",
        y="max_ambiguity",
        hue="stable",
        style="frontier",
        size="added_columns",
        sizes=(80, 320),
        ax=ax,
    )
    ax.axhline(
        ambiguity_limit,
        color="#C44E52",
        linestyle="--",
        linewidth=1.5,
        label="review limit",
    )
    ax.set_title("Public representation frontier")
    ax.set_xlabel("Public cells")
    ax.set_ylabel("Worst ambiguity")
    plt.show()
    if frontier.minimal_stable is not None:
        print(f"minimal stable representation: {frontier.minimal_stable.label}")
    else:
        print("no evaluated representation meets the ambiguity limit")